Q1): Dataset Splitting Strategy Task: You are building a lazy-loading "recipe" generator for the 10 genres.

Instead of loading audio into RAM, you will create a list of dictionaries. Assume you create exactly 100 recipe dictionaries per genre. Pass this combined list of recipes into sklearn.model_selection.train_test_split using test_size=0.2, shuffle=True, and random_state=42.

Question: Exactly how many recipe items will be allocated to your validation set (val_recipes) list?

**Answer:** If there's 100 recipe dictionaries per genre, that's 1000 in total. If `test_size` is set to `0.2`, then there will be `1000 * 0.2 = 200` recipes in the val set.

Q2) On-the-Fly Mixing Dimensions Task: In your training loop, you will load audio stems on the fly. 

Assume you use librosa.load(sr=16000, duration=10) to load 4 instrument stems and 1 noise file. You pad or truncate all 5 arrays to be exactly 160,000 samples long. 

You sum the 4 stems together to create a mix array, and then add the noise array scaled by an intensity weight of 0.2. 


Question: Before this final mix array is passed to the feature extractor, what must its exact 1D NumPy shape be? 

In [19]:
import librosa
from src.audio import pad_or_trim

s1, _ = librosa.load("data/messy_mashup/genres_stems/blues/blues.00000/bass.wav", sr=16000, duration=10)
s2, _ = librosa.load("data/messy_mashup/genres_stems/blues/blues.00001/drums.wav", sr=16000, duration=10)
s3, _ = librosa.load("data/messy_mashup/genres_stems/blues/blues.00002/other.wav", sr=16000, duration=10)
s4, _ = librosa.load("data/messy_mashup/genres_stems/blues/blues.00003/vocals.wav", sr=16000, duration=10)
noise, _ = librosa.load("data/messy_mashup/ESC-50-master/audio/1-137-A-32.wav", sr=16000, duration=10)
noise = pad_or_trim(noise, sr=16000, target_duration=10)

In [26]:
import numpy as np

mix = np.sum([s1, s2, s3, s4], axis=0)
noisy_mix = np.sum([mix, noise * 0.2], axis=0)
noisy_mix.shape

(160000,)

Question 3: Hugging Face Feature Extractor Shape Task: 


Initialize Hugging Face's AutoFeatureExtractor using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

To test your pipeline, create a dummy audio mix of exactly 160,000 ones using 
mix = np.ones(160000). 
Pass this dummy array into the extractor with sampling_rate=16000 and return_tensors="pt". 
Finally, extract the input_values from the resulting dictionary and apply .squeeze(0) to remove the default batch dimension. 


Question: What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process). In this format: [a,b] or (a,b)

In [ ]:
from transformers import AutoFeatureExtractor

extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
mix = np.ones(160000)
mix_in = extractor(mix, sampling_rate=16000, return_tensors="pt")

In [9]:
mix_in["input_values"].squeeze(0).shape

torch.Size([1024, 128])

Q4: Model Architecture Initialization Task: 
Initialize the ASTForAudioClassification.from_pretrained model using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

Because we are classifying 10 genres instead of the original AudioSet classes, you must explicitly pass num_labels=10 and ignore_mismatched_sizes=True to replace the classification head. 

Question: Write a quick loop using 

sum(p.numel() for p in model.parameters() if p.requires_grad) 

to calculate the number of trainable parameters in this newly configured model. What is the exact integer value?


In [10]:
from transformers import AutoModelForAudioClassification

model = AutoModelForAudioClassification.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593", ignore_mismatched_sizes=True, num_labels=10)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [11]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

86196490

Q5): Inference Normalization Math Task: In your test inference loop, right before feature extraction, the audio is normalized to prevent clipping using this exact formula: 

y = y / (np.max(np.abs(y)) + 1e-9). 

To test this, create a NumPy array: 
y_test = np.array([-0.85, 0.40, 0.20, -0.10]). 

Apply the normalization formula to it. 

Question: What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?

In [14]:
y_test = np.array([-0.85, 0.4, 0.2, -0.1])

y_test_norm = y_test / np.max(np.abs(y_test) + 1e-9)

np.round(y_test_norm[0], 3)

np.float64(-1.0)